In [1]:
# Cell 1: Install
!pip install -q transformers datasets accelerate peft
print("Done")

Done


In [2]:
# Cell 2: Load model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, gc, json, requests

MODEL_NAME = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
DEVICE = next(model.parameters()).device

print(f"Loaded {MODEL_NAME} on {DEVICE}")
print(f"  Layers: {model.config.n_layer}")
print(f"  d_model: {model.config.n_embd}")
print(f"  Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded gpt2-xl on cuda:0
  Layers: 48
  d_model: 1600
  Free GPU: 20.3 GB


In [3]:
# Cell 3: Load data
cf = requests.get("https://rome.baulab.info/data/dsets/counterfact.json").json()
print(f"CounterFact: {len(cf)} samples")

from datasets import load_dataset
mmlu_ds = load_dataset("cais/mmlu", "all", split="test")
mmlu_sample = list(mmlu_ds.select(range(200)))
print(f"MMLU: {len(mmlu_sample)} questions")

CounterFact: 21919 samples


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

MMLU: 200 questions


In [4]:
# Cell 4: Helpers
def get_prob(m, prompt, target_str):
    target_id = tokenizer.encode(" " + target_str.strip(), add_special_tokens=False)[0]
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        logits = m(**inputs).logits[0, -1, :]
    return torch.softmax(logits, dim=-1)[target_id].item()

def generate(m, prompt, max_new=15):
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def mmlu_accuracy(m, questions, n=200):
    correct = 0
    choices_map = ["A", "B", "C", "D"]
    for q in questions[:n]:
        prompt = q["question"] + "\n" + "".join(
            f"{c}. {ch}\n" for c, ch in zip(choices_map, q["choices"])
        ) + "Answer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
        with torch.no_grad():
            logits = m(**inputs).logits[0, -1, :]
        choice_ids = [tokenizer.encode(f" {c}", add_special_tokens=False)[0] for c in choices_map]
        pred = choice_ids[torch.argmax(torch.tensor([logits[i] for i in choice_ids])).item()]
        if pred == choice_ids[q["answer"]]:
            correct += 1
    return correct / min(n, len(questions))

def safe_mean(lst):
    vals = [x for x in lst if x is not None]
    return sum(vals) / len(vals) if vals else None

print("Helpers defined")

Helpers defined


In [5]:
# Cell 5: MMLU baseline
print("Computing MMLU baseline...")
mmlu_baseline = mmlu_accuracy(model, mmlu_sample, n=200)
print(f"MMLU baseline: {mmlu_baseline:.4f}")

Computing MMLU baseline...
MMLU baseline: 0.2550


In [6]:
# Cell 6: Gradient test — verify AlphaEdit will work on GPT-2-XL
for param in model.parameters():
    param.requires_grad_(True)
model.train()

test_text = "The mother tongue of Danielle Darrieux is French"
inputs_test = tokenizer(test_text, return_tensors="pt").to(model.device)
labels_test = inputs_test["input_ids"].clone()
labels_test[0, :-1] = -100

with torch.enable_grad():
    out = model(**inputs_test, labels=labels_test)
    print("Loss:", out.loss.item())
    print("Loss requires_grad:", out.loss.requires_grad)
    print("grad_fn:", out.loss.grad_fn)
    out.loss.backward()

param0 = next(p for p in model.parameters() if p.requires_grad)
print("Param grad is not None:", param0.grad is not None)

for param in model.parameters():
    param.requires_grad_(False)
model.eval()
torch.cuda.empty_cache()
print("Gradient test done")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Loss: 1.3784946203231812
Loss requires_grad: True
grad_fn: <NllLossBackward0 object at 0x7d003c0e8070>
Param grad is not None: True
Gradient test done


In [7]:
# Cell 7a: Constants
ALPHAEDIT_LAYER  = 17
ALPHAEDIT_LR     = 5.0
ALPHAEDIT_STEPS  = 50
ALPHAEDIT_MODULE = "transformer.h.{}.mlp.c_proj"

def get_module_gpt2(m, layer):
    mod = m
    for part in ALPHAEDIT_MODULE.format(layer).split("."):
        mod = getattr(mod, part)
    return mod

print("get_module_gpt2 defined")

get_module_gpt2 defined


In [8]:
# Cell 7b: Preservation inputs + projector
def collect_preservation_inputs_gpt2(m, prompts, layer, n=10):
    """Collect input activations to c_proj. Shape: [n, 6400]"""
    inputs_list = []
    target_mod  = get_module_gpt2(m, layer)
    hook = target_mod.register_forward_pre_hook(
        lambda mod, inp: inputs_list.append(inp[0][0, -1, :].detach().float().cpu())
    )
    for p in prompts[:n]:
        inp = tokenizer(p, return_tensors="pt").to(m.device)
        with torch.no_grad():
            m(**inp)
    hook.remove()
    return torch.stack(inputs_list)  # [n, 6400]

def compute_null_space_projector(C):
    """C: [n, 6400]. Returns P: [6400, 6400] on CPU."""
    C       = C.cpu().float()
    CCT     = C @ C.T
    CCT_inv = torch.linalg.pinv(CCT)
    P       = torch.eye(C.shape[1], dtype=torch.float32) - C.T @ CCT_inv @ C
    return P

print("collect_preservation_inputs_gpt2 and compute_null_space_projector defined")

collect_preservation_inputs_gpt2 and compute_null_space_projector defined


In [9]:
# Cell 7c: Shape sanity check
tm = get_module_gpt2(model, ALPHAEDIT_LAYER)
W  = tm.weight  # [6400, 1600]

sample0      = cf[0]
nbr0         = sample0.get("neighborhood_prompts", [])
pres0        = nbr0[:5] if nbr0 else [sample0["requested_rewrite"]["prompt"].format(sample0["requested_rewrite"]["subject"])]
C0           = collect_preservation_inputs_gpt2(model, pres0, ALPHAEDIT_LAYER)
P0           = compute_null_space_projector(C0)
delta0       = torch.zeros_like(W.data.float())  # [6400, 1600]

print(f"W shape:        {W.shape}")        # [6400, 1600]
print(f"C shape:        {C0.shape}")       # [n, 6400]
print(f"P shape:        {P0.shape}")       # [6400, 6400]
print(f"delta shape:    {delta0.shape}")   # [6400, 1600]
proj = P0.to(delta0.device) @ delta0
print(f"P @ delta shape:{proj.shape}")     # [6400, 1600] ✓
del C0, P0, delta0, proj
torch.cuda.empty_cache()
print("Shapes correct")

W shape:        torch.Size([6400, 1600])
C shape:        torch.Size([5, 6400])
P shape:        torch.Size([6400, 6400])
delta shape:    torch.Size([6400, 1600])
P @ delta shape:torch.Size([6400, 1600])
Shapes correct


In [20]:
def run_alphaedit_gpt2(base_model, sample, layer=ALPHAEDIT_LAYER, steps=ALPHAEDIT_STEPS):
    rw          = sample["requested_rewrite"]
    prompt      = rw["prompt"].format(rw["subject"])
    target_new  = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    base_model.eval()
    baseline_p = get_prob(base_model, prompt, target_new)

    nbr_prompts  = sample.get("neighborhood_prompts", [])
    pres_prompts = nbr_prompts[:10] if nbr_prompts else [prompt]

    C      = collect_preservation_inputs_gpt2(base_model, pres_prompts, layer)
    P      = compute_null_space_projector(C).to(base_model.device)  # [6400, 6400]
    tm     = get_module_gpt2(base_model, layer)
    W_orig = tm.weight.data.clone().float()  # [6400, 1600]
    b_orig = tm.bias.data.clone().float()    # [1600]

    train_text = prompt + " " + target_new.strip()
    inputs     = tokenizer(train_text, return_tensors="pt").to(base_model.device)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
    labels     = inputs["input_ids"].clone()
    labels[0, :prompt_len] = -100

    delta     = torch.zeros_like(W_orig, requires_grad=True)  # [6400, 1600]
    optimizer = torch.optim.Adam([delta], lr=ALPHAEDIT_LR)

    def conv1d_hook(module, input, output):
        # Conv1D does: output = x @ weight + bias
        # x: [batch, seq, 6400], weight: [6400, 1600], bias: [1600]
        # We replace weight with W_orig + P@delta (all in graph)
        x           = input[0].float()
        delta_proj  = P.to(delta.dtype) @ delta        # [6400, 1600]
        effective_W = W_orig.to(x.device) + delta_proj # [6400, 1600]
        bias        = b_orig.to(x.device, dtype=x.dtype)
        # Reshape x for addmm: [batch*seq, 6400]
        size_out = x.size()[:-1] + (effective_W.size(1),)
        out = torch.addmm(bias, x.reshape(-1, x.size(-1)),
                          effective_W.to(x.dtype))
        return out.view(size_out).to(output.dtype)

    for param in base_model.parameters():
        param.requires_grad_(True)
    base_model.train()

    for step in range(steps):
        optimizer.zero_grad()
        hook = tm.register_forward_hook(conv1d_hook)
        with torch.enable_grad():
            out  = base_model(**inputs, labels=labels)
            loss = out.loss
        hook.remove()
        if step == 0:
            gn = delta.grad.norm().item() if delta.grad is not None else "NO GRAD"
            print(f"  step 0: loss={loss.item():.4f} delta_grad={gn}")
        loss.backward()
        optimizer.step()

    for param in base_model.parameters():
        param.requires_grad_(False)

    # Apply final edit
    with torch.no_grad():
        delta_proj = P.to(delta.dtype) @ delta
        tm.weight.data = (W_orig + delta_proj).to(tm.weight.dtype)

    base_model.eval()

    with torch.no_grad():
        p_after           = get_prob(base_model, prompt, target_new)
        edit_success      = p_after
        gen_after         = generate(base_model, prompt, max_new=15)
        edit_success_bool = target_new.lower().strip() in gen_after.lower()

        para_prompts = sample.get("paraphrase_prompts", [])
        para_success = safe_mean(
            [get_prob(base_model, p, target_new) for p in para_prompts[:5]]
        ) if para_prompts else None

        if nbr_prompts:
            bleed_probs, pres_probs = [], []
            base_correct = base_correct_broken = 0
            for p in nbr_prompts[:5]:
                tm.weight.data = W_orig.to(tm.weight.dtype)
                base_p_true = get_prob(base_model, p, target_true)
                tm.weight.data = (W_orig + delta_proj.detach()).to(tm.weight.dtype)
                edit_p_new  = get_prob(base_model, p, target_new)
                edit_p_true = get_prob(base_model, p, target_true)
                bleed_probs.append(edit_p_new)
                pres_probs.append(edit_p_true)
                if base_p_true > 0.05:
                    base_correct += 1
                    if edit_p_true < base_p_true * 0.5:
                        base_correct_broken += 1
            nbr_bleed        = sum(bleed_probs) / len(bleed_probs)
            nbr_preservation = sum(pres_probs)  / len(pres_probs)
            oe_damage        = base_correct_broken / base_correct if base_correct > 0 else 0.0
        else:
            nbr_bleed = nbr_preservation = oe_damage = None

    tm.weight.data = W_orig.to(tm.weight.dtype)
    del delta, optimizer, C, P
    torch.cuda.empty_cache(); gc.collect()

    return {
        "subject":                   rw["subject"],
        "prompt":                    prompt,
        "target_new":                target_new,
        "target_true":               target_true,
        "baseline_p":                baseline_p,
        "p_after":                   p_after,
        "edit_success":              edit_success,
        "edit_success_bool":         edit_success_bool,
        "paraphrase_success":        para_success,
        "neighborhood_bleed":        nbr_bleed,
        "neighborhood_preservation": nbr_preservation,
        "oe_damage":                 oe_damage,
        "locality_drop":             None,
        "gen_after":                 gen_after,
        "alphaedit_layer":           layer,
        "alphaedit_steps":           steps,
    }

print("run_alphaedit_gpt2() redefined — Conv1D forward hook approach")

run_alphaedit_gpt2() redefined — Conv1D forward hook approach


In [21]:
print("Smoke test cf[0] with 20 steps...")
r0 = run_alphaedit_gpt2(model, cf[0], steps=20)
print(f"  edit_p={r0['edit_success']:.4f}")
print(f"  gen:   {r0['gen_after']!r}")
print(f"  bleed: {r0['neighborhood_bleed']:.4f}")

Smoke test cf[0] with 20 steps...
  step 0: loss=3.4976 delta_grad=NO GRAD
  edit_p=0.9927
  gen:   'English Brooke Bryan Dunn Wyatt Forrest NXT Hollow Winston Macy Brooke Destiny Brooke Reason Brooke'
  bleed: 0.1101


In [22]:
# Step sweep on cf[:5] to find good step counts
for steps in [5, 10, 20, 50]:
    edit_ps = []
    for sample in cf[:5]:
        r = run_alphaedit_gpt2(model, sample, steps=steps)
        edit_ps.append(r['edit_success'])
    print(f"steps={steps:3d}  avg={sum(edit_ps)/len(edit_ps):.3f}  {[round(x,3) for x in edit_ps]}")

  step 0: loss=2.9200 delta_grad=NO GRAD
  step 0: loss=6.5392 delta_grad=NO GRAD
  step 0: loss=9.1024 delta_grad=NO GRAD
  step 0: loss=10.7261 delta_grad=NO GRAD
  step 0: loss=15.9652 delta_grad=NO GRAD
steps=  5  avg=0.869  [0.89, 0.631, 0.963, 0.861, 1.0]
  step 0: loss=2.4594 delta_grad=NO GRAD
  step 0: loss=5.5864 delta_grad=NO GRAD
  step 0: loss=8.7747 delta_grad=NO GRAD
  step 0: loss=12.6995 delta_grad=NO GRAD
  step 0: loss=15.4486 delta_grad=NO GRAD
steps= 10  avg=0.970  [0.939, 0.99, 0.982, 0.94, 1.0]
  step 0: loss=3.8550 delta_grad=NO GRAD
  step 0: loss=5.1428 delta_grad=NO GRAD
  step 0: loss=9.2751 delta_grad=NO GRAD
  step 0: loss=11.4670 delta_grad=NO GRAD
  step 0: loss=15.3541 delta_grad=NO GRAD
steps= 20  avg=0.991  [0.992, 0.995, 0.988, 0.979, 1.0]
  step 0: loss=2.8043 delta_grad=NO GRAD
  step 0: loss=6.1779 delta_grad=NO GRAD
  step 0: loss=8.9509 delta_grad=NO GRAD
  step 0: loss=11.1884 delta_grad=NO GRAD
  step 0: loss=16.4806 delta_grad=NO GRAD
steps= 

In [23]:
for steps in [5, 10, 20, 50]:
    edit_ps = []
    for sample in cf[10:15]:
        r = run_alphaedit_gpt2(model, sample, steps=steps)
        rw = sample["requested_rewrite"]
        print(f"  steps={steps} target={rw['target_new']['str']!r} edit_p={r['edit_success']:.3f}")
        edit_ps.append(r['edit_success'])
    print(f"steps={steps:3d}  avg={sum(edit_ps)/len(edit_ps):.3f}\n")

  step 0: loss=14.4152 delta_grad=NO GRAD
  steps=5 target='Sega' edit_p=0.954
  step 0: loss=9.6807 delta_grad=NO GRAD
  steps=5 target='football' edit_p=0.962
  step 0: loss=9.9796 delta_grad=NO GRAD
  steps=5 target='Russian' edit_p=0.997
  step 0: loss=10.4969 delta_grad=NO GRAD
  steps=5 target='Microsoft' edit_p=0.998
  step 0: loss=6.9214 delta_grad=NO GRAD
  steps=5 target='French' edit_p=0.233
steps=  5  avg=0.829

  step 0: loss=14.5356 delta_grad=NO GRAD
  steps=10 target='Sega' edit_p=0.993
  step 0: loss=8.6935 delta_grad=NO GRAD
  steps=10 target='football' edit_p=0.979
  step 0: loss=9.7578 delta_grad=NO GRAD
  steps=10 target='Russian' edit_p=0.994
  step 0: loss=10.1268 delta_grad=NO GRAD
  steps=10 target='Microsoft' edit_p=0.999
  step 0: loss=6.7586 delta_grad=NO GRAD
  steps=10 target='French' edit_p=0.442
steps= 10  avg=0.881

  step 0: loss=13.6390 delta_grad=NO GRAD
  steps=20 target='Sega' edit_p=0.998
  step 0: loss=10.6185 delta_grad=NO GRAD
  steps=20 target

In [24]:
STEP_OPTIONS = [(5, 1), (20, 5), (50, 10)]

all_results_ae   = {}
all_summaries_ae = {}

for STEPS, N_STEPS in STEP_OPTIONS:
    print(f"\n{'='*55}")
    print(f"  AlphaEdit — {STEPS} steps (n_steps={N_STEPS}) — GPT-2-XL")
    print(f"{'='*55}\n")

    results = []
    for i, sample in enumerate(cf[:50]):
        try:
            res = run_alphaedit_gpt2(model, sample, steps=STEPS)
            res["idx"] = i
            results.append(res)
            b = f"{res['neighborhood_bleed']:.3f}" if res['neighborhood_bleed'] is not None else "N/A"
            p = f"{res['paraphrase_success']:.3f}" if res['paraphrase_success'] is not None else "N/A"
            print(f"[{i:02d}] edit_p={res['edit_success']:.3f} bleed={b} para={p} | {res['gen_after'][:40]!r}")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print(f"[{i:02d}] OOM")
            results.append({"idx": i, "error": "OOM", "edit_success": 0.0})
        except Exception as e:
            torch.cuda.empty_cache(); gc.collect()
            print(f"[{i:02d}] ERROR: {e}")
            results.append({"idx": i, "error": str(e), "edit_success": 0.0})

    print("Computing MMLU locality drop...")
    mmlu_post     = mmlu_accuracy(model, mmlu_sample, n=200)
    locality_drop = round(mmlu_baseline - mmlu_post, 4)
    print(f"  baseline={mmlu_baseline:.3f} post={mmlu_post:.3f} drop={locality_drop:.4f}")

    good = [r for r in results if "error" not in r]
    summary = {
        "method":    f"AlphaEdit_{N_STEPS}step_gpt2xl",
        "model":     "gpt2-xl",
        "n_samples": len(good),
        "metrics": {
            "edit_success":              safe_mean([r["edit_success"] for r in good]),
            "paraphrase_success":        safe_mean([r.get("paraphrase_success") for r in good]),
            "neighborhood_bleed":        safe_mean([r.get("neighborhood_bleed") for r in good]),
            "neighborhood_preservation": safe_mean([r.get("neighborhood_preservation") for r in good]),
            "oe_damage":                 safe_mean([r.get("oe_damage") for r in good]),
            "locality_drop":             locality_drop,
        },
        "samples": results,
    }
    all_results_ae[STEPS]   = results
    all_summaries_ae[STEPS] = summary

    out_path = f"/content/alphaedit_{N_STEPS}step_gpt2xl.json"
    with open(out_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\n  Edit:     {summary['metrics']['edit_success']:.1%}")
    print(f"  Bleed:    {summary['metrics']['neighborhood_bleed']:.1%}")
    print(f"  Para:     {summary['metrics']['paraphrase_success']:.1%}")
    print(f"  Locality: {locality_drop:.4f}")
    print(f"  Saved: {out_path}")
    from google.colab import files
    files.download(out_path)

print(f"\n\n{'='*55}")
print("FINAL SUMMARY")
print(f"{'='*55}")
print(f"{'Steps':<8} {'n_steps':<10} {'Edit':>8} {'Bleed':>8} {'Para':>8} {'Locality':>10}")
print("-" * 55)
for STEPS, N_STEPS in STEP_OPTIONS:
    m = all_summaries_ae[STEPS]["metrics"]
    print(f"{STEPS:<8} {N_STEPS:<10} {m['edit_success']:>8.1%} "
          f"{m['neighborhood_bleed']:>8.1%} {m['paraphrase_success']:>8.1%} "
          f"{m['locality_drop']:>10.4f}")


  AlphaEdit — 5 steps (n_steps=1) — GPT-2-XL

  step 0: loss=3.1202 delta_grad=NO GRAD
[00] edit_p=0.981 bleed=0.005 para=0.022 | 'English Bryan Stall Compton Ingram Foste'
  step 0: loss=6.1469 delta_grad=NO GRAD
[01] edit_p=0.949 bleed=0.000 para=0.000 | 'Islam Yus Mub Tunis Emirisal Amin Faw Al'
  step 0: loss=9.1684 delta_grad=NO GRAD
[02] edit_p=0.908 bleed=0.929 para=0.333 | 'piano Piano Orchestra Roma Bohem Sakura '
  step 0: loss=11.5189 delta_grad=NO GRAD
[03] edit_p=0.959 bleed=0.128 para=0.956 | 'Swedenenegger Kron Rasmquist mailing tro'
  step 0: loss=16.0193 delta_grad=NO GRAD
[04] edit_p=1.000 bleed=0.002 para=0.000 | 'Manila conquasaridium Hud EminSpawn Bubq'
  step 0: loss=3.1666 delta_grad=NO GRAD
[05] edit_p=0.623 bleed=0.006 para=0.006 | 'English Ally Fleming Wire LD Straight Al'
  step 0: loss=9.9130 delta_grad=NO GRAD
[06] edit_p=0.887 bleed=0.005 para=0.744 | 'PhiladelphiaBoston Cecilganszinotten Amb'
  step 0: loss=4.6901 delta_grad=NO GRAD
[07] edit_p=0.937 ble

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  AlphaEdit — 20 steps (n_steps=5) — GPT-2-XL

  step 0: loss=3.0689 delta_grad=NO GRAD
[00] edit_p=0.994 bleed=0.052 para=0.009 | 'English Wallace Boyd Neal Compton Mitche'
  step 0: loss=7.7600 delta_grad=NO GRAD
[01] edit_p=0.988 bleed=0.000 para=0.001 | 'Islam Axis Jihadrent Mub terroristenemy '
  step 0: loss=8.9680 delta_grad=NO GRAD
[02] edit_p=0.976 bleed=0.380 para=0.484 | 'piano Piano Guitar Melody Rhythm pian al'
  step 0: loss=10.8850 delta_grad=NO GRAD
[03] edit_p=0.975 bleed=0.001 para=0.974 | 'Sweden debianquistAk EditorsOPAJJtaboola'
  step 0: loss=16.7324 delta_grad=NO GRAD
[04] edit_p=1.000 bleed=0.001 para=0.006 | 'Manila Baal CarbuncleSourceFiledaq\x1cFontS'
  step 0: loss=3.5085 delta_grad=NO GRAD
[05] edit_p=0.974 bleed=0.004 para=0.000 | 'English hobbies Levin WRITE Enterprise 5'
  step 0: loss=9.8144 delta_grad=NO GRAD
[06] edit_p=0.968 bleed=0.005 para=0.898 | 'Philadelphiaingeneller Hartfordelin Fisc'
  step 0: loss=4.3271 delta_grad=NO GRAD
[07] edit_p=0.982

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  AlphaEdit — 50 steps (n_steps=10) — GPT-2-XL

  step 0: loss=2.9333 delta_grad=NO GRAD
[00] edit_p=0.998 bleed=0.000 para=0.014 | 'English Bryan Dickinson Derby steroids W'
  step 0: loss=4.8658 delta_grad=NO GRAD
[01] edit_p=0.995 bleed=0.000 para=0.000 | 'Islam Mubisal サーティワン Mub Mub Mub Mub Mub'
  step 0: loss=9.1837 delta_grad=NO GRAD
[02] edit_p=0.996 bleed=0.300 para=0.457 | 'piano Rhythm Piano Rih Melody Kyoto Orch'
  step 0: loss=11.1786 delta_grad=NO GRAD
[03] edit_p=0.990 bleed=0.004 para=0.992 | 'Swedenudiudiocoleneggerudiudiudiudiudiud'
  step 0: loss=15.3062 delta_grad=NO GRAD
[04] edit_p=1.000 bleed=0.021 para=0.347 | 'Manila foldersCVESourceFileffiti occupan'
  step 0: loss=3.5956 delta_grad=NO GRAD
[05] edit_p=0.998 bleed=0.000 para=0.000 | 'EnglishOX Staples HGOVERrawdownloadOSED\x0f'
  step 0: loss=9.7811 delta_grad=NO GRAD
[06] edit_p=0.991 bleed=0.038 para=0.961 | 'Philadelphiaiola intentional Theoeller F'
  step 0: loss=4.4859 delta_grad=NO GRAD
[07] edit_p=0.98

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



FINAL SUMMARY
Steps    n_steps        Edit    Bleed     Para   Locality
-------------------------------------------------------
5        1             84.5%    10.6%    28.9%     0.0000
20       5             98.3%     7.5%    40.2%     0.0000
50       10            97.8%     9.3%    40.8%     0.0000


##Reload dont run

In [15]:
# Reload model to fix corrupted weights
del model
torch.cuda.empty_cache()
gc.collect()

from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    "gpt2-xl", torch_dtype=torch.float16, device_map="auto"
)
model.eval()
print("Model reloaded")

# Now check both MLP modules
sample_input = tokenizer("The mother tongue of Danielle Darrieux is",
                          return_tensors="pt").to(model.device)

for name in ["mlp.c_fc", "mlp.c_proj"]:
    mod = model
    for part in f"transformer.h.17.{name}".split("."):
        mod = getattr(mod, part)
    acts = []
    hook = mod.register_forward_pre_hook(
        lambda m, inp: acts.append(inp[0].shape)
    )
    with torch.no_grad():
        model(**sample_input)
    hook.remove()
    print(f"{name}: weight={mod.weight.shape}, input={acts[0]}")

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model reloaded
mlp.c_fc: weight=torch.Size([1600, 6400]), input=torch.Size([1, 9, 1600])
mlp.c_proj: weight=torch.Size([6400, 1600]), input=torch.Size([1, 9, 6400])
